In [1]:
# import importlib.util
# from pathlib import Path

# def _import_uatr_path():
#     seeds = []
#     p = globals().get("__vsc_ipynb_file__")
#     if isinstance(p, str):
#         seeds.append(Path(p).resolve().parent)
#     cwd = Path.cwd().resolve()
#     seeds.extend([cwd, *cwd.parents])
#     for base in seeds:
#         for cand in (
#             base / "_uatr_path.py",
#             base / "dev" / "research_nbs" / "_uatr_path.py",
#         ):
#             if cand.is_file():
#                 return cand
#     raise RuntimeError(
#         "Cannot find _uatr_path.py. Save/pull the repo and open the UATR folder in the editor."
#     )

# _spec = importlib.util.spec_from_file_location("_uatr_path", _import_uatr_path())
# _up = importlib.util.module_from_spec(_spec)
# assert _spec.loader is not None
# _spec.loader.exec_module(_up)
# _up.ensure_repo_on_sys_path()

import utils as ut
from check_circle_v2 import transition_matrix_circularity

In [2]:
# imports and settings

import os
import time
import pickle
import warnings
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from copy import deepcopy

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

%load_ext autoreload
%autoreload 2

# do not show warnings
warnings.filterwarnings("ignore")

print("Imports complete.")

Imports complete.


In [3]:
path = '/Users/itamarmerfeld/Projects/UATR/data/ds2/pos/dpv1_1m.wav'
fs, data = wavfile.read(path)

fft_nperseg=16000
percent_overlap=0.25
window='hann'
remove_dc=20
crop_freq=2000
normalization_window_size=9
Q=100

F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc, crop_freq=crop_freq)
pxx = ut.calc_welch_from_spectrogram(Sxx, normalization_window_size=normalization_window_size)
circularity_score = []
circularity_period = []
circularity_leading_eigenvalue = []
circularity_eigenvalues = []
circularity_spectral_concentration = []

# for fi in range(len(F)):
#     M = ut.get_s2g(phase[fi, :], n_levels=Q)
#     circularity = transition_matrix_circularity(M)
#     circularity_score.append(circularity.score)
#     circularity_period.append(circularity.period)
#     circularity_leading_eigenvalue.append(circularity.leading_eigenvalue)
#     circularity_eigenvalues.append(circularity.eigenvalues)
#     circularity_spectral_concentration.append(circularity.spectral_concentration)

fig = make_subplots(rows=1, cols=2, column_widths=[0.8, 0.2], column_titles=["Spectrogram", "Spectrum"], shared_yaxes=True)
fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='Viridis', showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=pxx, y=F, mode='lines', name='Normalized Spectrum', marker_color='Blue', showlegend=False), row=1, col=2)
# fig.add_trace(go.Scatter(x=circularity_score, y=F, mode='lines', name='Circularity Score', marker_color='Red', showlegend=False), row=1, col=3)
fig.update_layout(height=600, width=1000, font=dict(size=20))
fig.update_xaxes(title_text='Time (s)', title_font=dict(size=20), col=1, row=1)
fig.update_yaxes(title_text='Frequency (Hz)', title_font=dict(size=20), col=1, row=1)
fig.update_xaxes(title_text='Norm. Power', title_font=dict(size=20), col=2, row=1)
fig.update_yaxes(title_text='Frequency (Hz)', title_font=dict(size=20), col=2, row=1)
for ann in fig.layout.annotations:
    ann.font = dict(size=20)
fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/frequency_analysis.png", scale=10)
fig.show()

In [4]:
F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)
f_list = [636, 588, 800, 2000]
Q = 100

specs = [
    [{"type": "polar"} for _ in f_list],   # Row 1: Barpolar (phase distribution)
    [{"type": "xy"}    for _ in f_list],   # Row 2: Bar (transition-size / delta phase)
    [{"type": "xy"}    for _ in f_list],   # Row 3: Heatmap (transition matrix)
    [{"type": "xy"}    for _ in f_list],   # Row 4: graph (S2G)
]

fig = make_subplots(
    rows=4, cols=len(f_list),
    column_titles=[f"f={f} Hz" for f in f_list],
    row_titles=["Phase Dist.", "Trans. Size Dist.", "S2G Graph", "Trans. Dist."],
    specs=specs,
)

for ix, f in enumerate(f_list, start=1):
    f_ix = np.argmin(np.abs(F - f))
    f_phase = phase[f_ix, :]
    f_phase_wrapped = np.mod(f_phase, 2 * np.pi)   # np.angle returns [-pi, pi]; shift to [0, 2pi]

    # phase distribution (polar)
    hist_phase, bin_edges_phase = np.histogram(f_phase_wrapped, bins=Q, range=(0, 2 * np.pi), density=True)
    bin_centers = 0.5 * (bin_edges_phase[:-1] + bin_edges_phase[1:])
    bin_width = bin_edges_phase[1] - bin_edges_phase[0]

    fig.add_trace(go.Barpolar(
        r=hist_phase,
        theta=np.degrees(bin_centers),   # Plotly's polar theta is in degrees
        width=np.degrees(bin_width),
        marker_color=hist_phase,
        marker_colorscale='Viridis',
        showlegend=False,
    ), row=1, col=ix)

    # distribution of delta phases (transition size)
    delta_phase = np.abs(np.diff(f_phase))
    hist_delta, bin_edges_delta = np.histogram(delta_phase, bins=Q, range=(0, 2 * np.pi), density=True)
    fig.add_trace(go.Bar(
        x=bin_edges_delta[:-1], y=hist_delta,
        name='Delta Phase Distribution', marker_color='orange', showlegend=False,
    ), row=2, col=ix)

    # graph from S2G
    G = ut.get_s2g_graph(f_phase, n_levels=Q)
    fig_g = ut.draw_graph(G)
    for tr in fig_g.data:
        fig.add_trace(deepcopy(tr), row=3, col=ix)

    # transition matrix
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=4, col=ix)

fig.update_polars(
    angularaxis=dict(
        direction='counterclockwise',
        rotation=0,
        tickmode='array',
        tickvals=[0, 45, 90, 135, 180, 225, 270, 315],
        ticktext=['0', 'π/4', 'π/2', '3π/4', 'π', '5π/4', '3π/2', '7π/4'],
        tickfont=dict(size=20),
    ),
    radialaxis=dict(showticklabels=False),
)
fig.update_layout(
    height=1200, width=1200,
    font=dict(size=20),
    title_font=dict(size=20),
    margin=dict(t=140, r=140),   # leave room for lifted column titles + shifted row titles
)
fig.update_xaxes(tickfont=dict(size=20), title_font=dict(size=20))
fig.update_yaxes(tickfont=dict(size=20), title_font=dict(size=20))

# Match column/row title offsets and bold them.
# Column titles sit above row 1 at y = 1 + offset; row titles sit to the
# right of the last column at x = 1 + offset (in paper coords).
title_offset = 0.06
column_title_texts = {f"f={f} Hz" for f in f_list}
row_title_texts = {"Phase Dist.", "Trans. Size Dist.", "S2G Graph", "Trans. Dist."}
for ann in fig.layout.annotations:
    ann.font = dict(size=20)
    if ann.text in column_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.y = 1 + title_offset
        ann.yanchor = 'bottom'
    elif ann.text in row_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.x = 1 + title_offset
        ann.xanchor = 'left'
# fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/phase_distribution_analysis.pdf")
fig.show()

In [5]:
f0 = 1000
fs = 8000
duration = 60
snr = 0
Q = 10

sim_signal = ut.simulate_raw_signal(f0=f0, fs=fs, duration=duration)
noisy_sim_signal = ut.add_noise_to_signal(signal=sim_signal, snr_db=snr, fs=fs, signal_bw=1)
wn = np.random.randn(len(sim_signal))
signals = [sim_signal, noisy_sim_signal, wn]
column_titles = ["Original", "Noisy", "White Noise"]
row_titles = ["Phase Dist.", "S2G Graph", "Trans. Dist."]

specs = [
    [{"type": "polar"} for _ in signals],   # Row 1: Barpolar (phase distribution)
    [{"type": "xy"}    for _ in signals],   # Row 2: graph (S2G)
    [{"type": "xy"}    for _ in signals],   # Row 3: Heatmap (transition matrix)
]

fig = make_subplots(
    rows=3, cols=len(signals),
    column_titles=column_titles,
    row_titles=row_titles,
    specs=specs,
)

for ix, sig in enumerate(signals, start=1):
    F, T, Sxx, phase = ut.calc_spectrogram(sig, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)

    # phase distribution (polar) - pick the bin closest to f0
    f_ix = np.argmin(np.abs(F - f0))
    f_phase = phase[f_ix, :]
    f_phase_wrapped = np.mod(f_phase, 2 * np.pi)   # np.angle returns [-pi, pi]; shift to [0, 2pi]
    hist_phase, bin_edges_phase = np.histogram(f_phase_wrapped, bins=Q, range=(0, 2 * np.pi), density=True)
    bin_centers = 0.5 * (bin_edges_phase[:-1] + bin_edges_phase[1:])
    bin_width = bin_edges_phase[1] - bin_edges_phase[0]

    fig.add_trace(go.Barpolar(
        r=hist_phase,
        theta=np.degrees(bin_centers),   # Plotly's polar theta is in degrees
        width=np.degrees(bin_width),
        marker_color=hist_phase,
        marker_colorscale='Viridis',
        showlegend=False,
    ), row=1, col=ix)

    # graph from S2G
    G = ut.get_s2g_graph(f_phase, n_levels=Q)
    fig_g = ut.draw_graph(G)
    for tr in fig_g.data:
        fig.add_trace(deepcopy(tr), row=2, col=ix)

    # transition matrix
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=3, col=ix)

fig.update_polars(
    angularaxis=dict(
        direction='counterclockwise',
        rotation=0,
        tickmode='array',
        tickvals=[0, 45, 90, 135, 180, 225, 270, 315],
        ticktext=['0', 'π/4', 'π/2', '3π/4', 'π', '5π/4', '3π/2', '7π/4'],
        tickfont=dict(size=20),
    ),
    radialaxis=dict(showticklabels=False),
)
fig.update_layout(
    height=1200, width=1200,
    font=dict(size=20),
    title_font=dict(size=20),
    margin=dict(t=140, r=140),   # leave room for lifted column titles + shifted row titles
)
fig.update_xaxes(tickfont=dict(size=20), title_font=dict(size=20))
fig.update_yaxes(tickfont=dict(size=20), title_font=dict(size=20))

# Match column/row title offsets and bold them.
title_offset = 0.06
column_title_texts = set(column_titles)
row_title_texts = set(row_titles)
for ann in fig.layout.annotations:
    ann.font = dict(size=20)
    if ann.text in column_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.y = 1 + title_offset
        ann.yanchor = 'bottom'
    elif ann.text in row_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.x = 1 + title_offset
        ann.xanchor = 'left'
# fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/phase_distribution_analysis_signals.pdf")
fig.show()

In [6]:
F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)
f_list = [636, 588, 800, 2000]
Q = 10

specs = [
    [{"type": "xy"}    for _ in f_list],   # Row 2: Bar (transition-size / delta phase)
    [{"type": "xy"}    for _ in f_list],   # Row 3: Heatmap (transition matrix)
    [{"type": "xy"}    for _ in f_list],   # Row 4: graph (S2G)
]

fig = make_subplots(
    rows=3, cols=len(f_list),
    column_titles=[f"f={f} Hz" for f in f_list],
    row_titles=["Trans. Size Dist.", "S2G Graph", "Trans. Dist."],
    specs=specs,
)

for ix, f in enumerate(f_list, start=1):
    f_ix = np.argmin(np.abs(F - f))
    f_phase = phase[f_ix, :]
    f_phase_wrapped = np.mod(f_phase, 2 * np.pi)   # np.angle returns [-pi, pi]; shift to [0, 2pi]

    # distribution of delta phases (transition size)
    delta_phase = np.abs(np.diff(f_phase))
    hist_delta, bin_edges_delta = np.histogram(delta_phase, bins=Q, range=(0, 2 * np.pi), density=True)
    fig.add_trace(go.Bar(
        x=bin_edges_delta[:-1], y=hist_delta,
        name='Delta Phase Distribution', marker_color='orange', showlegend=False,
    ), row=1, col=ix)

    # graph from S2G
    G = ut.get_s2g_graph(f_phase, n_levels=Q)
    fig_g = ut.draw_graph(G)
    for tr in fig_g.data:
        fig.add_trace(deepcopy(tr), row=2, col=ix)

    # transition matrix
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=3, col=ix)

fig.update_layout(
    height=900, width=1200,
    font=dict(size=20),
    title_font=dict(size=20),
    margin=dict(t=140, r=140),   # leave room for lifted column titles + shifted row titles
)
fig.update_xaxes(tickfont=dict(size=20), title_font=dict(size=20))
fig.update_yaxes(tickfont=dict(size=20), title_font=dict(size=20))

# Match column/row title offsets and bold them.
# Column titles sit above row 1 at y = 1 + offset; row titles sit to the
# right of the last column at x = 1 + offset (in paper coords).
title_offset = 0.06
column_title_texts = {f"f={f} Hz" for f in f_list}
row_title_texts = {"Trans. Size Dist.", "S2G Graph", "Trans. Dist."}
for ann in fig.layout.annotations:
    ann.font = dict(size=20)
    if ann.text in column_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.y = 1 + title_offset
        ann.yanchor = 'bottom'
    elif ann.text in row_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.x = 1 + title_offset
        ann.xanchor = 'left'
fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/phase_distribution_analysis_part_2.png", scale=10)
fig.show()

In [7]:
F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)
f_list = [636, 588, 800, 2000]
Q = 10

fig = make_subplots(
    rows=1, cols=len(f_list),
    column_titles=[f"f={f} Hz" for f in f_list],
)

for ix, f in enumerate(f_list, start=1):
    f_ix = np.argmin(np.abs(F - f))
    f_phase = phase[f_ix, :]
    f_phase_wrapped = np.mod(f_phase, 2 * np.pi)   # np.angle returns [-pi, pi]; shift to [0, 2pi]

    # transition matrix
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=1, col=ix)

fig.update_layout(
    height=500, width=1500,
    font=dict(size=20),
    title_font=dict(size=20),
    margin=dict(t=140, r=140),   # leave room for lifted column titles + shifted row titles
)
fig.update_xaxes(tickfont=dict(size=20), title_font=dict(size=20))
fig.update_yaxes(tickfont=dict(size=20), title_font=dict(size=20))

# Match column/row title offsets and bold them.
# Column titles sit above row 1 at y = 1 + offset; row titles sit to the
# right of the last column at x = 1 + offset (in paper coords).
title_offset = 0.06
column_title_texts = {f"f={f} Hz" for f in f_list}
row_title_texts = {"Trans. Dist."}
for ann in fig.layout.annotations:
    ann.font = dict(size=20)
    if ann.text in column_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.y = 1 + title_offset
        ann.yanchor = 'bottom'
    elif ann.text in row_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.x = 1 + title_offset
        ann.xanchor = 'left'
fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/phase_distribution_analysis_part_3.png", scale=10)
fig.show()

In [ ]:
f0 = 1000
fs = 8000
duration = 60
snr = 0

sim_signal = ut.simulate_raw_signal(f0=f0, fs=fs, duration=duration)
noisy_sim_signal = ut.add_noise_to_signal(signal=sim_signal, snr_db=snr, fs=fs, signal_bw=1)
wn = np.random.rand(len(sim_signal))
signals = [sim_signal, noisy_sim_signal, wn]

fig = make_subplots(rows=1, cols=3, column_widths=[0.3, 0.3, 0.4], column_titles=["Original", "Noisy", "White Noise"])

for ix, signal in enumerate(signals, start=1):
    F, T, Sxx, phase = ut.calc_spectrogram(signal, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)

    # spectrogram
    fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='Viridis', showscale=False), row=1, col=ix)

    # phase distribution
    f_ix = np.argmin(np.abs(F - f0))
    f_phase = phase[f_ix, :]
    f_phase_wrapped = np.mod(f_phase, 2 * np.pi)   # np.angle returns [-pi, pi]; shift to [0, 2pi]
    


fig.update_layout(
    height=500, width=1500,
    font=dict(size=20),
    title_font=dict(size=20),
    margin=dict(t=140, r=140),   # leave room for lifted column titles + shifted row titles
)
fig.update_xaxes(tickfont=dict(size=20), title_font=dict(size=20))
fig.update_yaxes(tickfont=dict(size=20), title_font=dict(size=20))



In [7]:
f0 = 636
fs = 128000
duration = 60
snr = 0
Q = 10
percent_overlap = 0.5
fft_nperseg = 8192
window = 'hann'
remove_dc = 20

sim_signal = ut.simulate_raw_signal(f0=f0, fs=fs, duration=duration)
noisy_sim_signal = ut.add_noise_to_signal(signal=sim_signal, snr_db=snr, fs=fs, signal_bw=1)
wn = np.random.randn(len(sim_signal))

# feature scores:
laplacian_scores = []
wasserstein_scores = []

### Plotting
signals = [sim_signal, noisy_sim_signal, wn]
column_titles = ["Pure Tonal", "Noisy (SNR=0 dB)", "White Noise"]
row_titles = ["S2G Graph", "Trans. Dist."]  # ["Phase Dist.", "S2G Graph", "Trans. Dist."]

specs = [
    # [{"type": "polar"} for _ in signals],   # Row 1: Barpolar (phase distribution)
    [{"type": "xy"}    for _ in signals],   # Row 2: graph (S2G)
    [{"type": "xy"}    for _ in signals],   # Row 3: Heatmap (transition matrix)
]

fig = make_subplots(
    rows=2, cols=len(signals),
    column_titles=column_titles,
    row_titles=row_titles,
    specs=specs,
    vertical_spacing=0.1
)

for ix, sig in enumerate(signals, start=1):
    F, T, Sxx, phase = ut.calc_spectrogram(sig, fs, nperseg=8192, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)

    # phase distribution (polar) - pick the bin closest to f0
    f_ix = np.argmin(np.abs(F - f0))
    f_phase = phase[f_ix, :]
    f_phase_wrapped = np.mod(f_phase, 2 * np.pi)   # np.angle returns [-pi, pi]; shift to [0, 2pi]
    hist_phase, bin_edges_phase = np.histogram(f_phase_wrapped, bins=Q, range=(0, 2 * np.pi), density=True)
    bin_centers = 0.5 * (bin_edges_phase[:-1] + bin_edges_phase[1:])
    bin_width = bin_edges_phase[1] - bin_edges_phase[0]

    # fig.add_trace(go.Barpolar(
    #     r=hist_phase,
    #     theta=np.degrees(bin_centers),   # Plotly's polar theta is in degrees
    #     width=np.degrees(bin_width),
    #     marker_color=hist_phase,
    #     marker_colorscale='Viridis',
    #     showlegend=False,
    # ), row=1, col=ix)

    # graph from S2G
    G = ut.get_s2g_graph(f_phase, n_levels=Q)
    fig_g = ut.draw_graph(G)
    for tr in fig_g.data:
        fig.add_trace(deepcopy(tr), row=1, col=ix)

    # transition matrix
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    laplacian = ut.get_K(x_transitions, mode='laplacian')
    wasserstein = ut.get_K(x_transitions, mode='wasserstein')
    laplacian_scores.append(laplacian)
    wasserstein_scores.append(wasserstein)

    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=2, col=ix)

# fig.update_polars(
#     angularaxis=dict(
#         direction='counterclockwise',
#         rotation=0,
#         tickmode='array',
#         tickvals=[0, 45, 90, 135, 180, 225, 270, 315],
#         ticktext=['0', 'π/4', 'π/2', '3π/4', 'π', '5π/4', '3π/2', '7π/4'],
#         tickfont=dict(size=20),
#     ),
#     radialaxis=dict(showticklabels=False),
# )
fig.update_layout(
    height=800, width=1200,
    font=dict(size=20),
    title_font=dict(size=20),
    margin=dict(t=140, r=140, b=100),   # leave room for titles + feature subtitles below row 2
)
fig.update_xaxes(tickfont=dict(size=20), title_font=dict(size=20))
fig.update_yaxes(tickfont=dict(size=20), title_font=dict(size=20))

# Add subtitle with features for each column (below 2nd row)
for ix, (lap, wass) in enumerate(zip(laplacian_scores, wasserstein_scores), start=1):
    fig.add_annotation(
        text=f"Laplacian: {lap:.2f}<br>Wasserstein: {wass:.2f}",
        x=0.5, y=-0.18,
        xref="x domain", yref="y domain",
        showarrow=False,
        yanchor="top",
        row=2, col=ix,
    )

# Match column/row title offsets and bold them.
title_offset = 0.06
column_title_texts = set(column_titles)
row_title_texts = set(row_titles)
for ann in fig.layout.annotations:
    ann.font = dict(size=20)
    if ann.text in column_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.y = 1 + title_offset
        ann.yanchor = 'bottom'
    elif ann.text in row_title_texts:
        ann.text = f"<b>{ann.text}</b>"
        ann.x = 1 + title_offset
        ann.xanchor = 'left'
# fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/graph_examples.png", scale=10)
fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/graph_examples.pdf")
fig.show()